In [6]:
import pandas as pd
import torch
from alibi_detect.cd import MMDDrift
from pathlib import Path
import numpy as np

# ImageNet vs ImageNet-R

In [7]:
# Carica i dati
baseline = pd.read_csv("../results/ImageNetR/baseline_features.csv")
current = pd.read_csv("../results/ImageNetR/drift_features.csv")
REPORT_PATH_R = "../results/ImageNetR/mmd_drift_report.csv"
# Converte in numpy float32 (richiesto da alibi-detect)
X_ref = baseline.values.astype("float32")
X_cur = current.values.astype("float32")


In [8]:
# Inizializza il detector, backend pytorch userà la GPU se disponibile
cd = MMDDrift(
    X_ref,
    backend="pytorch",
    p_val=0.05,          # soglia di significatività
    n_permutations=100,  # numero di permutazioni per il test (più alto = più preciso ma più lento)
    device="cuda"         # forza esplicitamente la GPU; puoi ometterlo e lo fa in automatico
)

# Esegui il test
result = cd.predict(X_cur, return_p_val=True, return_distance=True)
data = result["data"]

In [9]:
report = pd.DataFrame([{
    "dataset": "ImageNetR",
    "n_reference": X_ref.shape[0],
    "n_target": X_cur.shape[0],
    "is_drift": bool(data["is_drift"]),
    "p_val": float(data["p_val"]),
    "distance": float(data["distance"]),
}])

report.to_csv(REPORT_PATH_R, index=False)
print(report.to_string(index=False))

  dataset  n_reference  n_target  is_drift  p_val  distance
ImageNetR        10000     10000      True    0.0  0.080451


# ImageNet vs ImageNet-C

In [5]:


BASE_DIR = Path("../results/ImageNetC")
BASELINE_PATH = BASE_DIR / "baseline_features.csv"
REPORT_PATH = BASE_DIR / "mmd_drift_report.csv"
N_REF = 10000
N_TARGET = 10000
SEED = 42
LABEL_COL = "label"  # metti "label" o "class" se esiste
def sample_features(csv_path: Path, n: int, label_col=None, seed=42):
    df = pd.read_csv(csv_path)

    if label_col is not None and label_col in df.columns:
        n_per_class = max(1, n // df[label_col].nunique())
        sampled = (
            df.groupby(label_col, group_keys=False)
              .apply(
                  lambda g: g.sample(min(len(g), n_per_class), random_state=seed),
                  include_groups=False
              )
              .reset_index(drop=True)
        )
        if len(sampled) > n:
            sampled = sampled.sample(n=n, random_state=seed).reset_index(drop=True)
    else:
        sampled = df.sample(n=min(len(df), n), random_state=seed).reset_index(drop=True)

    X = sampled.drop(columns=[label_col], errors="ignore").values.astype("float32")
    return X
def parse_corruption(path: Path):
    # Es: blur/defocus_blur_sev_1_features.csv
    rel = path.relative_to(BASE_DIR)
    rel_str = rel.as_posix()
    # togli _features.csv
    name = rel_str.replace("_features.csv", "")
    return name



In [6]:
# Reference: ImageNet clean
X_ref = sample_features(
    "../results/ImageNetC/baseline_features.csv",
    n=10000,
    label_col="label",  # metti "label" se presente
    seed=42
)


# Detector
device = "cuda" if torch.cuda.is_available() else "cpu"

cd = MMDDrift(
    X_ref,
    backend="pytorch",
    p_val=0.05,
    n_permutations=20,   # basso per test veloci
    device=device
)


In [9]:
results = []
# Entra in tutte le sottocartelle
for target_file in sorted(BASE_DIR.rglob("*_features.csv")):
    if target_file.name == BASELINE_PATH.name:
        continue

    X_target = sample_features(target_file, N_TARGET, label_col=LABEL_COL, seed=SEED)

    detector = MMDDrift(
        X_ref,
        backend="pytorch",
        p_val=0.05,
        n_permutations=20,
        device=device
    )

    pred = detector.predict(X_target, return_distance=True)
    data = pred.get("data", {})

    p_val = data.get("p_val", np.nan)
    distance = data.get("distance", np.nan)
    is_drift = bool(data.get("is_drift", False))

    corr_name = parse_corruption(target_file)

    results.append({
        "corruption": corr_name,
        "category": corr_name.split("/")[0] if "/" in corr_name else "root",
        "severity": corr_name.split("_sev_")[-1] if "_sev_" in corr_name else "NA",
        "is_drift": is_drift,
        "p_val": float(p_val) if not pd.isna(p_val) else np.nan,
        "distance": float(distance) if not pd.isna(distance) else np.nan,
        "device": device,
        "n_reference": X_ref.shape[0],
        "n_target": X_target.shape[0]
    })

    print(f"{corr_name:>40} | drift={is_drift} | p_val={p_val} | distance={distance}")

if not results:
    raise RuntimeError(f"Nessun file *_features.csv trovato sotto {BASE_DIR}")

report = pd.DataFrame(results).sort_values("p_val", na_position="last")
report.to_csv(REPORT_PATH, index=False)

print("\nCSV salvato in:", REPORT_PATH)
print(report.to_string(index=False))

                 blur/defocus_blur_sev_1 | drift=True | p_val=0.0 | distance=0.00021606683731079102
                 blur/defocus_blur_sev_2 | drift=True | p_val=0.0 | distance=0.0003153681755065918
                 blur/defocus_blur_sev_3 | drift=True | p_val=0.0 | distance=0.0004977583885192871
                 blur/defocus_blur_sev_4 | drift=True | p_val=0.0 | distance=0.0009176135063171387
                 blur/defocus_blur_sev_5 | drift=True | p_val=0.0 | distance=0.0022034049034118652
                   blur/glass_blur_sev_1 | drift=True | p_val=0.0 | distance=0.000217437744140625
                   blur/glass_blur_sev_2 | drift=True | p_val=0.0 | distance=0.003066539764404297
                   blur/glass_blur_sev_3 | drift=True | p_val=0.0 | distance=0.017571866512298584
                   blur/glass_blur_sev_4 | drift=True | p_val=0.0 | distance=0.03211551904678345
                   blur/glass_blur_sev_5 | drift=True | p_val=0.0 | distance=0.02384793758392334
                